### 1. ????? import

In [ ]:
import pandas as pd # pandas? ???? ??? ?? ?????
from sklearn.model_selection import train_test_split  # train_test_split? ???? train? test? ??? ?? ?????
from sklearn.preprocessing import StandardScaler  # StandardScaler? ???? ????? ?? ?????
from sklearn.linear_model import LogisticRegression # LogisticRegression? ???? ??? ?? ?????
from sklearn.ensemble import RandomForestClassifier # RandomForestClassifier? ?? ????? ?? ?????
from xgboost import XGBClassifier # XGBClassifier? XGBoost? ?? ?????
from sklearn.metrics import classification_report, accuracy_score # classification_report? accuracy_score? import


### 2. data load and Split

In [ ]:
df_df = pd.read_csv("player_TopRate_position_JJINMAK/DF_combined.csv")
df_df.columns # ???? ??? ??

### 3. VIF ??

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

def calculate_vif(dataframe):
    """
    ??? ??????? VIF(?? ?? ??)? ??.

    Args:
        dataframe (pd.DataFrame): ?? ???? ???? ??????.

    Returns:
        pd.DataFrame: ?? ??? ?? VIF ?.
    """
    # ??? ?? (???? ??? ???? ??)
    X = add_constant(dataframe)
    
    # VIF ??
    vif_data = pd.DataFrame()
    vif_data["Feature"] = X.columns
    vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

    return vif_data

df_df_drop = df_df.copy()
df_df_drop = df_df.drop(columns= ['??', "???_DF", "???_FW", "???_GK", "???_MF", "??", "isWin"])
# ?? ???
# df: ?? ???? ???? ??????
vif_result = calculate_vif(df_df_drop)
vif_result = vif_result[vif_result['VIF'] < 7].reset_index(drop=True)
# VIF ?? ??
print(vif_result)
feature_list = vif_result['Feature'].tolist()


In [ ]:
vif_result_sorted = vif_result.sort_values(by=vif_result.columns[1], ascending=False)

# ?? ??
vif_result_sorted

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# VIF ?? ?????? ??
vif_result_sorted = vif_result.sort_values(by='VIF', ascending=True)

# ???
plt.figure(figsize=(10, 6))
sns.barplot(x='VIF', y='Feature', data=vif_result_sorted, palette='viridis')
plt.title('VIF Values by Feature in DF')
plt.xlabel('VIF')
plt.ylabel('Feature')
plt.show()

In [ ]:
# '??'? ?????? '???'? ??? ??? ???, ??? ??? ??? ??
filtered_features = [
    feature for feature in feature_list if '???' in feature or '??' not in feature
]

print(len(filtered_features))

In [ ]:
X = df_df[filtered_features]
y = df_df["isWin"]  # ?? ??

# ??/?? ??? ??
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

### 4. LogisticRegression learning and evalutaion (RFE)
- RFE(Recursive Feature Elimination)

In [ ]:
from sklearn.feature_selection import RFE
# ?? ??
model = LogisticRegression(max_iter=10000)

# RFE? ?? ?? ??
selector = RFE(estimator=model, n_features_to_select=20)  # ??? ?? ? ??
selector = selector.fit(X_train, y_train)

# ??? ?? ??
selected_features = X_train.columns[selector.support_]  # ??? ???
print("??? ???:", selected_features)

# ??? ??? ??? ?? ??
X_train_selected = selector.transform(X_train)
X_test_selected = selector.transform(X_test)

# ?? ??
model.fit(X_train_selected, y_train)

# ?? ?? ??
accuracy = model.score(X_test_selected, y_test)
print(f"?? ???: {accuracy:.4f}")

### 5. LogisticRegression learning and evalutaion 

In [ ]:
scaler = StandardScaler() # ???? ?? ?? ??
X_train_scaled = scaler.fit_transform(X_train) # ?? ???? ???
X_test_scaled = scaler.transform(X_test) # ?? ???? ???

In [ ]:
logistic_model = LogisticRegression(random_state=42) # ???? ?? ?? ??

In [ ]:
logistic_model.fit(X_train_scaled, y_train) # ?? ??
logistic_preds = logistic_model.predict(X_test_scaled) # ?? ??

In [ ]:
print("Logistic Regression Performance:") # ?? ??
print(classification_report(y_test, logistic_preds)) # ?? ??? ??
print("Accuracy:", accuracy_score(y_test, logistic_preds)) # ??? ??

### 6. RandomForest and evalutaion 

In [ ]:
rf_model = RandomForestClassifier(random_state=42, n_estimators=100, max_depth=10) # ?? ???? ?? ??

In [ ]:
rf_model.fit(X_train, y_train) # ?? ??
rf_preds = rf_model.predict(X_test) # ?? ??

In [ ]:
print("\nRandom Forest Performance:") # ?? ??
print(classification_report(y_test, rf_preds)) # ?? ??? ??
print("Accuracy:", accuracy_score(y_test, rf_preds)) # ??? ??

### 7. XGBoost and evalutaion 

In [ ]:
xgb_model = XGBClassifier(random_state=42, n_estimators=100, max_depth=10, learning_rate=0.1) # XGBoost ?? ??

In [ ]:
xgb_model.fit(X_train, y_train) # ?? ??
xgb_preds = xgb_model.predict(X_test) # ?? ??

In [ ]:
print("\nXGBoost Performance:") # ?? ??
print(classification_report(y_test, xgb_preds)) # ?? ??? ??
print("Accuracy:", accuracy_score(y_test, xgb_preds)) # ??? ??

### 8. Logistic, Random, XGBoost overfit check

In [ ]:
from sklearn.metrics import confusion_matrix # confusion_matrix? import

In [ ]:
# ?? ???
models = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42, n_estimators=100, max_depth=10),
    "XGBoost": XGBClassifier(random_state=42, n_estimators=100, max_depth=10, learning_rate=0.1),
}

# ?? ?? ?? ??
def evaluate_model(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    accuracy = (tp + tn) / (tn + fp + fn + tp)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return accuracy, precision, recall, f1

# ?? ?? ? ??
for model_name, model in models.items():
    print(f"\nTraining {model_name}...")
    
    # ??
    model.fit(X_train, y_train)
    
    # ?? ??? ??
    train_preds = model.predict(X_train)
    train_accuracy, train_precision, train_recall, train_f1 = evaluate_model(y_train, train_preds)
    
    # ??? ??? ??
    test_preds = model.predict(X_test)
    test_accuracy, test_precision, test_recall, test_f1 = evaluate_model(y_test, test_preds)
    
    # ?? ?? ??
    print(f"{model_name} Performance:")
    print(f"Train Accuracy: {train_accuracy:.4f}, Test Accuracy: {test_accuracy:.4f}")
    print(f"Train Precision: {train_precision:.4f}, Test Precision: {test_precision:.4f}")
    print(f"Train Recall: {train_recall:.4f}, Test Recall: {test_recall:.4f}")
    print(f"Train F1 Score: {train_f1:.4f}, Test F1 Score: {test_f1:.4f}")
    
    # ??? ?? ??
    if train_accuracy - test_accuracy > 0.1 or train_f1 - test_f1 > 0.1:
        print("Potential Overfitting Detected!")
    else:
        print("No Significant Overfitting Observed.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

### 9. Visualization

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


# ?? ??
models = {
    "Logistic Regression": LogisticRegression(max_iter=10000),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(eval_metric='logloss', random_state=42)
}

# ?? ??? ?? ?? ???
for model_name, model in models.items():
    print(f"??: {model_name}")
    
    # ?? ??: 5-fold ?? ???? ??? ??
    cv_scores = cross_val_score(model, X_train, y_train, cv=5)
    print(f"?? ?? ???: {cv_scores.mean():.4f} ? {cv_scores.std():.4f}")
    
    # ?? ??
    model.fit(X_train, y_train)
    
    # ??? ???? ??
    y_pred = model.predict(X_test)
    
    # ?? ?? ??
    cm = confusion_matrix(y_test, y_pred)
    
    # ?? ?? ???
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model.classes_)
    disp.plot(cmap=plt.cm.Blues)
    plt.title(f"{model_name} Confusion Matrix")
    plt.show()
    
    print("="*50)

# 10. XAI

In [ ]:
import shap
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split

# ??? ??
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# ?? ??
models = {
    "Logistic Regression": LogisticRegression(max_iter=10000),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
}

# SHAP ? ?? ? ???
for model_name, model in models.items():
    # ?? ??
    model.fit(X_train, y_train)
    
    # SHAP ??? ??
    explainer = shap.TreeExplainer(model) if model_name == "XGBoost" or model_name == "Random Forest" else shap.LinearExplainer(model, X_train)
    
    # SHAP ? ??
    shap_values = explainer.shap_values(X_test)
    
    # SHAP summary plot
    print(f"{model_name} SHAP Summary Plot")
    shap.summary_plot(shap_values, X_test)
    plt.show()
    
    # SHAP dependence plot (?? ??? ?? SHAP ?? ?? ???)
    # shap.dependence_plot(0, shap_values, X_test)  # 0? ?? (? ?? ??)? ?? ??? ??
    # plt.show()

In [ ]:
from lime.lime_tabular import LimeTabularExplainer
import numpy as np

# ?? ?? (??: ?? ????)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# LIME ??? ??
explainer = LimeTabularExplainer(X_train.values, training_labels=y_train.values, mode='classification', 
                                 feature_names=X.columns, class_names=['0', '1'], discretize_continuous=True)

# ?? ??? ?? ?? ??
i = 1  # ??? ??? ?? ???
explanation = explainer.explain_instance(X_test.iloc[i].values, model.predict_proba)

# LIME ?? ???
explanation.show_in_notebook(show_table=True, show_all=False)